# TAIHLS Biomedical Capstone — Starter Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/curriculum-v2/notebooks/06-capstone-starter.ipynb)

Welcome to your end-to-end **TAIHLS Capstone Project** (Chapter 5.6)! Everything below runs 100% offline with lightweight scientific Python (`numpy`, `pandas`, `matplotlib`, `scikit-learn`).

Pick **one** of three authentic biomedical tracks:
- **Track A — Oncology Triage:** `load_breast_cancer` (569 real biopsies, 30 nuclear measurements)
- **Track B — Diabetes Progression & Equity:** `load_diabetes(scaled=False)` (442 real patients, clinical units)
- **Track C — Genomic Sequence Classifier & Shift Stress Test:** Engineer GC + 2-mer features on promoter vs. coding sequences and test shortcut/shift robustness!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score, recall_score

BLUE, VERMILION, TEAL = '#0072B2', '#D55E00', '#009E73'
print('Setup ready!')

## Milestone 1 — Clinical / Biological Framing (15%)

**TODO:** Set `TRACK` to `'A'` or `'B'` (or replace with your Track C sequence feature matrix), state your exact question, define class `1`, and explain why False Negatives and False Positives carry unequal clinical costs.

In [ ]:
TRACK = 'A'   # <-- Change to 'B' for Diabetes Equity Track!

if TRACK == 'A':
    raw = load_breast_cancer(as_frame=True)
    X = raw.data
    y = (raw.target == 0).astype(int)      # 1 = malignant
    subgroup_series = X['mean radius'] >= X['mean radius'].median()
    subgroup_names = {False: 'Smaller tumors', True: 'Larger tumors'}
elif TRACK == 'B':
    raw = load_diabetes(as_frame=True, scaled=False)
    X = raw.data
    y = (raw.target >= 180).astype(int)    # 1 = high 1-year progression
    subgroup_series = X['age'] >= 50
    subgroup_names = {False: 'Age < 50', True: 'Age >= 50'}

my_question = 'TODO: Write your specific clinical question and why FN vs. FP costs differ.'
print(f'Track {TRACK}: {X.shape[0]} patients, {X.shape[1]} features | Positive rate = {y.mean():.1%}')
print(my_question)

## Milestone 2 — Data Exploration & Quality Audit (20%)

**TODO:** Check missing values, inspect mean vs. median for two key features, and create one clean annotated plot.

In [ ]:
print('Total missing values:', int(X.isna().sum().sum()))
col = X.columns[0]
print(f'{col}: mean={X[col].mean():.2f}, median={X[col].median():.2f}, min={X[col].min():.2f}, max={X[col].max():.2f}')

# TODO: Customize this plot to compare positive vs negative distributions!
fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.hist(X.loc[y == 0, col], bins=18, alpha=0.65, color=BLUE, label='Negative (0)')
ax.hist(X.loc[y == 1, col], bins=18, alpha=0.65, color=VERMILION, label='Positive (1)')
ax.set_xlabel(col); ax.set_ylabel('Patients'); ax.set_title(f'Distribution of {col} by outcome')
ax.legend(frameon=False); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## Milestone 3 — Leak-Free Modeling, Baselines & Clinical Thresholding (20%)

**TODO:** Compare Logistic Regression vs. Random Forest via 5-fold CV, pick a justified screening threshold `tau`, and compute held-out AUROC, AUPRC, Sensitivity, and Specificity.

In [ ]:
X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
    X, y, subgroup_series, test_size=0.25, random_state=0, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

baseline_acc = max(y_te.mean(), 1 - y_te.mean())
print(f'Majority-class baseline accuracy: {baseline_acc:.1%}')

candidates = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0),
}
for name, clf in candidates.items():
    cv_auc = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='roc_auc').mean()
    print(f'  {name:<22}: 5-fold CV AUROC = {cv_auc:.3f}')

# Fit your chosen model and evaluate ONCE on the locked test set
best_clf = candidates['Logistic Regression'].fit(X_tr, y_tr)
probs = best_clf.predict_proba(X_te)[:, 1]

tau = 0.30   # <-- TODO: Tune and justify your clinical decision threshold!
preds = (probs >= tau).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, preds, labels=[0, 1]).ravel()
print(f'\nHeld-out test @ tau={tau:.2f} | AUROC={roc_auc_score(y_te, probs):.3f} | '
      f'AUPRC={average_precision_score(y_te, probs):.3f}')
print(f'Sensitivity = {tp/(tp+fn):.1%} (FN={fn}) | Specificity = {tn/(tn+fp):.1%} (FP={fp})')

## Milestone 4 — Fairness, Bootstrap Uncertainty & Abstention Audit (25%)

**TODO:** Audit subgroup sensitivity gap, compute a 95% bootstrap CI for AUROC, and evaluate selective abstention.

In [ ]:
# 1. Subgroup audit
for val, label in subgroup_names.items():
    m = np.asarray(g_te == val)
    sens = recall_score(y_te[m], preds[m]) if y_te[m].sum() > 0 else np.nan
    print(f'{label:<16} (n={m.sum():>2}): sensitivity = {sens:.1%}')

# 2. TODO: Compute a 95% bootstrap confidence interval for AUROC (see Exercise 1.5)
rng = np.random.default_rng(0)
boot_aucs = []
yt_arr = np.asarray(y_te)
for _ in range(1000):
    idx = rng.integers(0, len(yt_arr), len(yt_arr))
    if len(np.unique(yt_arr[idx])) == 2:
        boot_aucs.append(roc_auc_score(yt_arr[idx], probs[idx]))
lo, hi = np.percentile(boot_aucs, [2.5, 97.5])
print(f'95% Bootstrap CI for AUROC: [{lo:.3f}, {hi:.3f}]')

# 3. Selective prediction / abstention at 85% confidence
conf = np.maximum(probs, 1 - probs)
keep = conf >= 0.85
auto_acc = ((probs[keep] >= 0.5).astype(int) == y_te[keep]).mean()
print(f'Abstention @ 85% conf: coverage = {keep.mean():.1%}, auto-accuracy = {auto_acc:.1%}, deferred = {(~keep).sum()}')

## Milestone 5 — Honest Health AI Model Card (20%)

**TODO:** Fill in all six sections honestly using the **8-Point Critical Appraisal Checklist** (Chapter 5.5):

- **Intended use & clinical role:** TODO
- **Training & evaluation data:** TODO
- **Primary performance (with baseline & 95% CI):** TODO
- **Subgroup fairness & calibration findings:** TODO
- **Known failure modes, shortcuts & privacy risks:** TODO
- **Human-in-the-loop deployment plan & needed clinical evidence:** TODO